# DCA na Queda - Compra intuitiva de BTC

Estratégia de Dollar-Cost Averaging (DCA) aplicada ao Bitcoin com três abordagens:

1. **Só na Queda**: Compra apenas quando BTC fecha em queda
2. **Todo Dia**: Compra um valor fixo todos os dias
3. **Todo Dia + Dobro na Queda** ⭐: Compra todo dia e dobra o aporte em quedas

## Setup

Instale as dependências:

In [ ]:
!pip install requests pandas numpy matplotlib

## Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## Parâmetros

In [ ]:
SYMBOL = 'BTCUSDT'
START = '2020-01-01'
BUY_USD = 10.0  # Aporte base em US$
USE_SYNTHETIC = False  # True para dados sintéticos de teste

## Funções

In [ ]:
def load_binance_daily(symbol='BTCUSDT', start='2020-01-01', end=None):
    '''Baixa candles diários da Binance'''
    import requests
    base = 'https://api.binance.com/api/v3/klines'
    start_ts = int(pd.Timestamp(start, tz='UTC').timestamp() * 1000)
    end_ts = int((pd.Timestamp(end, tz='UTC') if end
                  else pd.Timestamp.now(tz='UTC')).timestamp() * 1000)
    rows, cursor = [], start_ts
    while cursor < end_ts:
        r = requests.get(base, params={
            'symbol': symbol, 'interval': '1d',
            'startTime': cursor, 'limit': 1000}, timeout=15)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        rows.extend(batch)
        cursor = batch[-1][0] + 1
        time.sleep(0.25)
        if len(batch) < 1000:
            break
    df = pd.DataFrame(rows, columns=[
        'open_time','Open','High','Low','Close','Volume',
        'close_time','qav','trades','tbav','tqav','ignore'])
    df['time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    return df.set_index('time')[['Open','High','Low','Close','Volume']].astype(float)


def make_synthetic_daily(days=2000, seed=42):
    rng = np.random.default_rng(seed)
    idx = pd.date_range('2020-01-01', periods=days, freq='1D', tz='UTC')
    rets = rng.normal(0.0008, 0.035, days)
    close = 7000 * np.exp(np.cumsum(rets))
    df = pd.DataFrame({'Open': close, 'High': close, 'Low': close,
                       'Close': close, 'Volume': 1.0}, index=idx)
    return df

## Carregar dados

In [ ]:
if USE_SYNTHETIC:
    print('>> Dados sintéticos')
    daily = make_synthetic_daily()
else:
    print('>> Baixando diário da Binance...')
    daily = load_binance_daily(SYMBOL, START)

print(f'{len(daily)} dias | {daily.index[0].date()} a {daily.index[-1].date()}')
daily[['Close']].tail()

## Simulação

In [ ]:
def simular_dca(daily, buy_usd, estrategia='so_na_queda'):
    '''Simula DCA com diferentes estratégias'''
    px = daily['Close']
    caiu = px < px.shift(1)
    
    if estrategia == 'so_na_queda':
        comprar = caiu.copy()
        comprar.iloc[0] = False
        aporte_usd = buy_usd
    elif estrategia == 'todo_dia':
        comprar = pd.Series(True, index=px.index)
        comprar.iloc[0] = False
        aporte_usd = buy_usd
    elif estrategia == 'todo_dia_dobro_queda':
        comprar = pd.Series(True, index=px.index)
        comprar.iloc[0] = False
        aporte_usd = buy_usd * (2.0 * caiu + 1.0 * ~caiu).astype(float)
    else:
        raise ValueError(f'Estratégia desconhecida: {estrategia}')
    
    btc_comprado = (aporte_usd / px).where(comprar, 0.0)
    btc_acum = btc_comprado.cumsum()
    aportes = comprar.astype(float) * aporte_usd
    investido = aportes.cumsum()
    valor = btc_acum * px

    sim = pd.DataFrame({
        'preco': px,
        'comprou': comprar,
        'aporte_usd': aportes,
        'btc_comprado': btc_comprado,
        'btc_acum': btc_acum,
        'investido': investido,
        'valor': valor,
        'queda': caiu,
    })
    return sim


sim = simular_dca(daily, BUY_USD, estrategia='so_na_queda')
print('Simulação concluída')

## Resultados

In [ ]:
def resumo(sim, label):
    n_compras = int(sim['comprou'].sum())
    investido = sim['investido'].iloc[-1]
    btc = sim['btc_acum'].iloc[-1]
    preco_atual = sim['preco'].iloc[-1]
    valor = btc * preco_atual
    lucro = valor - investido
    roi = (valor / investido - 1) * 100 if investido else np.nan
    preco_medio = investido / btc if btc else np.nan
    return {
        'Estratégia': label,
        'Compras': n_compras,
        'Investido US$': investido,
        'BTC acumulado': btc,
        'Preço médio pago US$': preco_medio,
        'Preço atual US$': preco_atual,
        'Valor hoje US$': valor,
        'Lucro US$': lucro,
        'ROI %': roi,
    }


sim_todo_dia = simular_dca(daily, BUY_USD, estrategia='todo_dia')
sim_dobro_queda = simular_dca(daily, BUY_USD, estrategia='todo_dia_dobro_queda')

comp = pd.DataFrame([
    resumo(sim, 'Só na queda'),
    resumo(sim_todo_dia, 'Todo dia'),
    resumo(sim_dobro_queda, 'Todo dia + dobro na queda'),
]).set_index('Estratégia')

print('\n=== COMPARAÇÃO DAS ESTRATÉGIAS ===')
print(comp[['Compras','Investido US$','BTC acumulado','Preço médio pago US$',
      'Valor hoje US$','Lucro US$','ROI %']])

## Visualizações

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(sim.index, sim['valor'], lw=1.5, label='Só na queda', alpha=0.8)
ax.plot(sim_todo_dia.index, sim_todo_dia['valor'], lw=1.5, label='Todo dia', alpha=0.8)
ax.plot(sim_dobro_queda.index, sim_dobro_queda['valor'], lw=2.0, label='Todo dia + dobro na queda', 
        color='darkgreen', alpha=0.9)

ax.set_ylabel('Valor da carteira (US$)', fontsize=11)
ax.set_xlabel('Tempo', fontsize=11)
ax.set_title('Comparação de estratégias - Valor da carteira ao longo do tempo', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()